In [4]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

I load a ready-made sentiment model. It was already trained, so I only use it. This is called inference

In [7]:
from transformers import pipeline

clf = pipeline("sentiment-analysis",
               model="distilbert-base-uncased-finetuned-sst-2-english")
# print(clf(["This drug worked well for me", "The side effects were terrible"]))
print(clf(["It helped my pain but made me dizzy", "The medicine was not bad at all"]))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 581.52it/s]


[{'label': 'NEGATIVE', 'score': 0.9374473690986633}, {'label': 'POSITIVE', 'score': 0.9973673224449158}]


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)

text = "The medicine was not bad at all"
inputs = tokenizer(text, return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))  # step 1: pieces

with torch.no_grad():
    logits = model(**inputs).logits                             # step 2: raw scores
print(logits)

probs = torch.softmax(logits, dim=1)                            # step 3: probabilities
print(probs, model.config.id2label)                             # which number is which label

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 614.45it/s]

['[CLS]', 'the', 'medicine', 'was', 'not', 'bad', 'at', 'all', '[SEP]']
tensor([[-2.9076,  3.0295]])
tensor([[0.0026, 0.9974]]) {0: 'NEGATIVE', 1: 'POSITIVE'}


I can run the model many times by changing the input. I need this later to test if an explanation is honest

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    return {model.config.id2label[i]: round(probs[i].item(), 4) for i in range(2)}

print(predict("The medicine was almost bad at all"))
print(predict("i like the medicin"))   # the word "not" removed

{'NEGATIVE': 0.9995, 'POSITIVE': 0.0005}
{'NEGATIVE': 0.9997, 'POSITIVE': 0.0003}
